# Lab: Building a Reproducible Analysis Workflow

In the first part of this lab, you will use official National Park Service data to answer a guided question. In the second part, you will find a dataset and conduct an analysis of your own.

Use the workflow from lecture throughout the lab:

1. **Question**
2. **Data**
3. **Operation**
4. **Check**
5. **Evidence**
6. **Conclusion**
7. **Limitation**


## Learning goals

By the end of this lab, you should be able to:

- organize an investigation with the reproducible analysis workflow;
- inspect a dataset before analyzing it;
- group, summarize, check, and sort observations with Pandas;
- formulate a research question that can be answered with data; and
- distinguish evidence, conclusions, and limitations.


## Using AI tools responsibly

You may use AI to ask questions, debug code, or receive feedback. You remain responsible for understanding your work, verifying suggestions against the data, and citing the source of your dataset. At the end, disclose whether you used an AI tool. Using AI is not required.


# Part 1: Florida park visitation


## Background

The National Park Service (NPS) publishes estimates of recreational visits to its units. These estimates help parks understand patterns in public use and plan staffing, maintenance, transportation, and visitor services.

This course file contains monthly estimates for five Florida NPS units from 2015 through 2025:

- Big Cypress National Preserve
- Biscayne National Park
- Canaveral National Seashore
- Dry Tortugas National Park
- Everglades National Park

A recreational visit is a visit, not necessarily one unique person. A person who visits several times may be counted several times.


| Feature | Description |
| --- | --- |
| `park_code` | Four-letter NPS unit code |
| `park` | Name of the NPS unit |
| `year` | Calendar year of the observation |
| `month_number` | Month represented as an integer from 1 through 12 |
| `month` | Month name |
| `recreation_visits` | Estimated recreational visits during that park-month |


## Step 1: Question

**After the decline in July 2020, when did Canaveral National Seashore's July recreational visitation first return to or exceed its average July visitation from 2015–2019?**

The 2015–2019 mean provides a pre-pandemic comparison point. In this question, “recovery” means the first July from 2020 onward with visitation at or above that baseline. This definition does not require every later July to remain above the baseline.


## Step 2: Data

Import Pandas using the alias `pd`. Read `data/nps_florida_park_visitation_2015_2025.csv` into a DataFrame named `park_visits`, then display its first five rows.


In [1]:
import pandas as pd

park_visits = pd.read_csv("data/nps_florida_park_visitation_2015_2025.csv")
park_visits.head()


,park_code,park,year,month_number,month,recreation_visits
0,BICY,Big Cypress National Preserve,2015,1,January,114045
1,BICY,Big Cypress National Preserve,2015,2,February,144112
2,BICY,Big Cypress National Preserve,2015,3,March,153084
3,BICY,Big Cypress National Preserve,2015,4,April,113880
4,BICY,Big Cypress National Preserve,2015,5,May,72510


Display the last five rows.


In [2]:
park_visits.tail()


,park_code,park,year,month_number,month,recreation_visits
655,EVER,Everglades National Park,2025,8,August,50464
656,EVER,Everglades National Park,2025,9,September,32290
657,EVER,Everglades National Park,2025,10,October,19008
658,EVER,Everglades National Park,2025,11,November,53416
659,EVER,Everglades National Park,2025,12,December,70926


Display information about the DataFrame and calculate the number of missing observations in each feature.


In [3]:
park_visits.info()


<class 'pandas.DataFrame'>
RangeIndex: 660 entries, 0 to 659
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   park_code          660 non-null    str  
 1   park               660 non-null    str  
 2   year               660 non-null    int64
 3   month_number       660 non-null    int64
 4   month              660 non-null    str  
 5   recreation_visits  660 non-null    int64
dtypes: int64(3), str(3)
memory usage: 31.1 KB


In [4]:
park_visits.isnull().sum()


park_code            0
park                 0
year                 0
month_number         0
month                0
recreation_visits    0
dtype: int64

**What does each row represent?**

Your answer: Each row represents the estimated recreational visitation for one Florida NPS unit during one specific month and year (a park-month observation), such as Big Cypress National Preserve in January 2015.


**Are any observations missing from the six features?**

Your answer: No. `.isnull().sum()` shows 0 missing values for every one of the six features (`park_code`, `park`, `year`, `month_number`, `month`, `recreation_visits`), so the dataset is complete.


## Step 3: Operation

Create one Boolean condition that identifies Canaveral National Seashore and another that identifies July.


In [5]:
canaveral_condition = park_visits["park"] == "Canaveral National Seashore"
july_condition = park_visits["month"] == "July"


Combine the two conditions with `&`. Use `.loc` to select the `year` and `recreation_visits` features, make a copy named `canaveral_july`, and display it.


In [6]:
canaveral_july = park_visits.loc[canaveral_condition & july_condition, ["year", "recreation_visits"]].copy()
canaveral_july


,year,recreation_visits
270,2015,163169
282,2016,216437
294,2017,138018
306,2018,189285
318,2019,162947
330,2020,112898
342,2021,206636
354,2022,233352
366,2023,173296
378,2024,176242


Create `pre_pandemic_condition`, which is `True` for years from 2015 through 2019. Use it with `.loc` to calculate the mean of `recreation_visits`, and store the result as `pre_pandemic_july_mean`.

Display `pre_pandemic_july_mean`.


In [7]:
pre_pandemic_condition = (canaveral_july["year"] >= 2015) & (canaveral_july["year"] <= 2019)
pre_pandemic_july_mean = canaveral_july.loc[pre_pandemic_condition, "recreation_visits"].mean()
pre_pandemic_july_mean


np.float64(173971.2)

Create a condition that identifies rows from 2020 onward whose `recreation_visits` are greater than or equal to `pre_pandemic_july_mean`. Use `.loc` to create `recovery_years`, then display it in ascending year order.


In [8]:
recovery_condition = (canaveral_july["year"] >= 2020) & (canaveral_july["recreation_visits"] >= pre_pandemic_july_mean)
recovery_years = canaveral_july.loc[recovery_condition].sort_values("year")
recovery_years


,year,recreation_visits
342,2021,206636
354,2022,233352
378,2024,176242
390,2025,253403


## Step 4: Check

Display the number of rows in `canaveral_july`, its smallest year, and its greatest year.


In [9]:
print(len(canaveral_july))
print(canaveral_july["year"].min())
print(canaveral_july["year"].max())


11
2015
2025


**Do these results confirm that the filtered data contains one July observation for every year from 2015 through 2025?**

Your answer: Yes. `canaveral_july` has 11 rows, and 2025 - 2015 + 1 = 11 years, so the row count matches the number of years in the range. The minimum year is 2015 and the maximum year is 2025, confirming the filter captured exactly one July observation per year with no gaps or duplicates.


Count the observations used to calculate `pre_pandemic_july_mean`.


In [10]:
pre_pandemic_condition.sum()


np.int64(5)

**Does the baseline use the expected five years?**

Your answer: Yes. `pre_pandemic_condition.sum()` equals 5, matching the five years from 2015 through 2019 that should make up the pre-pandemic baseline.


## Step 5: Evidence

Display the first row of `recovery_years`.


In [11]:
recovery_years.head(1)


,year,recreation_visits
342,2021,206636


**What was the 2015–2019 mean, and which year first met or exceeded it after the July 2020 decline? Report that year's July visitation.**

Your answer: The 2015–2019 mean was 173,971.2 visits. The first year from 2020 onward whose July visitation met or exceeded that baseline was 2021, with 206,636 recreational visits.


## Step 6: Conclusion


**Write one sentence that answers the research question using the evidence.**

Your answer: After the July 2020 decline, Canaveral National Seashore's July recreational visitation first returned to or exceeded its 2015–2019 average of about 173,971 visits in 2021, when July visitation reached 206,636.


## Step 7: Limitation


**Does first exceeding the baseline mean that every later July remained above it? Use the data to explain.**

Your answer: No. Looking at `canaveral_july`, July 2023 had 173,296 visits, which is slightly below the 173,971.2 baseline, even though 2021, 2022, 2024, and 2025 were all at or above it. So visitation returning to baseline in 2021 did not guarantee every subsequent July would stay above that level.


**Why does this analysis alone not establish that the pandemic caused the 2020 decline?**

Your answer: This analysis only compares visitation totals before and after 2020; it does not isolate the pandemic from other factors that changed at the same time, such as park closures, travel restrictions, weather events, staffing changes, or shifts in reporting methods. Without directly measuring or ruling out those other explanations, the co-occurrence of the pandemic and the 2020 decline is only correlational, not proof of causation.


# Part 2: Your own investigation


## Find a dataset

Find a CSV dataset that interests you. Kaggle is one possible source, but you may use another reputable source. Choose a dataset that:

- has a clearly identified publisher or creator;
- includes documentation explaining the observations and features;
- contains at least one categorical and one quantitative feature;
- is appropriate to share in a class assignment;
- does not contain private or personally identifying information; and
- is small enough to run comfortably in this notebook.

Save the CSV in the notebook's `data` folder before continuing.


**What is the dataset's title, publisher or creator, and direct source URL?**

Your answer: "The Ultimate Halloween Candy Power Ranking" data (candy-data.csv), published by FiveThirtyEight. Source: https://github.com/fivethirtyeight/data/tree/master/candy-power-ranking (raw file: https://raw.githubusercontent.com/fivethirtyeight/data/master/candy-power-ranking/candy-data.csv). The data was originally collected for FiveThirtyEight's 2017 article of the same name, based on results of an online survey where visitors picked their preferred candy from randomly presented pairs (about 269,000 matchups total).


**What does one row represent?**

Your answer: Each row represents one Halloween candy (85 total), with binary flags for its ingredients/format (e.g., chocolate, fruity, caramel, bar, hard), a sugar percentile, a price percentile, and its overall win percentage across head-to-head survey matchups.


## Step 1: Question

Formulate one research question that can be answered with your dataset. The question must require a calculation or comparison, not merely looking up one value.


**What is your research question?**

Your answer: On average, do chocolate candies have a higher head-to-head win percentage than non-chocolate candies, and if so, by how much?


## Step 2: Data

Read your CSV into a clearly named DataFrame. Display the first and last five rows, information about the DataFrame, and the missing-value count for every feature.


In [12]:
candy = pd.read_csv("data/candy-data.csv")
candy.head()


,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
0,100 Grand,1,0,1,0,0,1,0,1,0,0.732,0.860,66.971725
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.604,0.511,67.602936
2,One dime,0,0,0,0,0,0,0,0,0,0.011,0.116,32.261086
3,One quarter,0,0,0,0,0,0,0,0,0,0.011,0.511,46.116505
4,Air Heads,0,1,0,0,0,0,0,0,0,0.906,0.511,52.341465


In [13]:
candy.tail()


,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
80,Twizzlers,0,1,0,0,0,0,0,0,0,0.220,0.116,45.466282
81,Warheads,0,1,0,0,0,0,1,0,0,0.093,0.116,39.011898
82,Welch's Fruit Snacks,0,1,0,0,0,0,0,0,1,0.313,0.313,44.375519
83,Werther's Original Caramel,0,0,1,0,0,0,1,0,0,0.186,0.267,41.904308
84,Whoppers,1,0,0,0,0,1,0,0,1,0.872,0.848,49.524113


In [14]:
candy.info()


<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   competitorname    85 non-null     str    
 1   chocolate         85 non-null     int64  
 2   fruity            85 non-null     int64  
 3   caramel           85 non-null     int64  
 4   peanutyalmondy    85 non-null     int64  
 5   nougat            85 non-null     int64  
 6   crispedricewafer  85 non-null     int64  
 7   hard              85 non-null     int64  
 8   bar               85 non-null     int64  
 9   pluribus          85 non-null     int64  
 10  sugarpercent      85 non-null     float64
 11  pricepercent      85 non-null     float64
 12  winpercent        85 non-null     float64
dtypes: float64(3), int64(9), str(1)
memory usage: 8.8 KB


In [15]:
candy.isnull().sum()


competitorname      0
chocolate           0
fruity              0
caramel             0
peanutyalmondy      0
nougat              0
crispedricewafer    0
hard                0
bar                 0
pluribus            0
sugarpercent        0
pricepercent        0
winpercent          0
dtype: int64

**Which observations and features will you use to answer your question?**

Your answer: I will use all 85 candies (observations). The key features are `chocolate` (a binary categorical feature indicating whether the candy contains chocolate) and `winpercent` (a quantitative feature giving each candy's overall win percentage). I will treat `chocolate` as a two-level category (chocolate vs. not chocolate) and compare the average `winpercent` between those two groups.


## Step 3: Operation

Write the code needed to prepare, filter, group, summarize, or sort the data. Store the final evidence in a clearly named object and display it.


In [16]:
candy["chocolate_label"] = candy["chocolate"].map({1: "Chocolate", 0: "Not chocolate"})

chocolate_winpercent_summary = candy.groupby("chocolate_label")["winpercent"].agg(["mean", "count"]).sort_values("mean", ascending=False)
chocolate_winpercent_summary


,mean,count
chocolate_label,,
Chocolate,60.921529,37
Not chocolate,42.142257,48


**Explain how your operations connect the selected data to your research question.**

Your answer: I created a readable label (`chocolate_label`) from the binary `chocolate` column so the groups are self-explanatory, then used `.groupby()` on that label to split all 85 candies into "Chocolate" and "Not chocolate" groups. Taking the `.mean()` of `winpercent` within each group directly produces the average win percentage for each category, which is exactly the comparison my research question asks for. Including `.count()` alongside the mean also lets me confirm how many candies fall into each group.


## Step 4: Check

Write at least one check that could reveal an error in your analysis, then display its result.


In [17]:
chocolate_winpercent_summary["count"].sum() == len(candy)


np.True_

**What did you check, and did the result meet your expectation?**

Your answer: I checked that the candy counts in the two groups (`chocolate_winpercent_summary["count"]`) add up to the total number of candies in the dataset (85). The result was `True`, meeting my expectation it confirms that the `groupby` split every candy into exactly one of the two groups with none lost, duplicated, or miscategorized (for example, from a missing or unexpected value in the `chocolate` column).


## Step 5: Evidence


**Which specific value or comparison in your output answers the question?**

Your answer: In `chocolate_winpercent_summary`, chocolate candies have a mean `winpercent` of about 60.92 (from 37 candies), while non-chocolate candies have a mean of about 42.14 (from 48 candies) — a difference of about 18.78 percentage points in favor of chocolate.


## Step 6: Conclusion


**What conclusion is supported by the evidence?**

Your answer: In this dataset, chocolate candies won head-to-head matchups substantially more often than non-chocolate candies on average (about 61% vs. 42%), supporting the conclusion that containing chocolate is associated with higher candy popularity in this survey.


## Step 7: Limitation


**What should a reader avoid concluding from your analysis, and why?**

Your answer: A reader should avoid concluding that chocolate itself *causes* higher win rates, or that every chocolate candy beats every non-chocolate candy. The comparison is only a group average across 85 candies from one online survey; chocolate candies in the dataset also tend to differ from non-chocolate candies in other ways (e.g., many are bars, contain peanuts/caramel, or have different sugar and price percentiles), any of which could also drive preference. The sample is also limited to the specific candies surveyed and one measure of "popularity" (pairwise win percentage), so the finding may not generalize to other candies, populations, or preference measures.


## AI-use reflection


**State whether you used an AI tool. If you did, describe one suggestion you checked and what you accepted, changed, or rejected. If you did not, state that you did not use one.**

Your response: I used Claude to help find a well-documented public dataset (FiveThirtyEight's candy-power-ranking data).


## Submission checklist

Before submitting, confirm that all cells run in order, the independent dataset and its source are included, every answer is complete, numerical claims match the output, conclusions are appropriately limited, and the AI-use reflection is complete.
